# Seminar 2. Custom PyTorch Operators

# Building Models in PyTorch Through Composition

PyTorch models are built using **composition**.  
Instead of defining one large monolithic network, we construct models by combining smaller, reusable modules.

Each module can contain other modules, which allows us to build hierarchical and well-structured architectures.

---

## Composition

Composition means:

- A model is built from smaller blocks.
- Each block can contain multiple layers.
- Blocks can be reused in larger architectures.
- Complex models are created by stacking simpler components.

This keeps code:

- Modular  
- Reusable  
- Readable  
- Easy to extend  


## Key Ideas

- Inherit from `nn.Module`
- Define layers inside `__init__`
- Define computation in `forward()`
- Create reusable blocks
- Build larger models by combining blocks

---

## Example: Model Built from Two Blocks

Below is a simple example where:

- We define a reusable blocks: `LinearReLUBlock` and `LinearTanhBlock`
- The final model is composed of two such blocks


In [8]:
import torch
import torch.nn as nn


class LinearReLUBlock(nn.Module):
    def __init__(self, in_features: int, out_features: int):
        super().__init__()

        self.linear = nn.Linear(in_features, out_features)
        self.activation = nn.ReLU()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.linear(x)
        x = self.activation(x)
        return x


class LinearTanhBlock(nn.Module):
    def __init__(self, in_features: int, out_features: int):
        super().__init__()

        self.linear = nn.Linear(in_features, out_features)
        self.activation = nn.Tanh()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.linear(x)
        x = self.activation(x)
        return x


class CombinedModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.block1 = LinearReLUBlock(4, 8)
        self.block2 = LinearTanhBlock(8, 8)
        self.output = nn.Linear(8, 2)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.block1(x)
        x = self.block2(x)
        x = self.output(x)
        return x


model = CombinedModel()
print(model)

CombinedModel(
  (block1): LinearReLUBlock(
    (linear): Linear(in_features=4, out_features=8, bias=True)
    (activation): ReLU()
  )
  (block2): LinearTanhBlock(
    (linear): Linear(in_features=8, out_features=8, bias=True)
    (activation): Tanh()
  )
  (output): Linear(in_features=8, out_features=2, bias=True)
)


# What `nn.Module` Enables

When we inherit from `nn.Module`, we automatically gain powerful functionality that works **recursively across all submodules**.

## What `nn.Module` Gives Us



### Parameter Registration

All layers assigned as attributes (e.g. `self.linear = nn.Linear(...)`) are:

- Automatically registered
- Collected in `model.parameters()`
- Included in `model.state_dict()`

This works **recursively** for all sub-blocks.

In [9]:
print("Registered parameters:")
for name, param in model.named_parameters():
    print(name, param.shape)


Registered parameters:
block1.linear.weight torch.Size([8, 4])
block1.linear.bias torch.Size([8])
block2.linear.weight torch.Size([8, 8])
block2.linear.bias torch.Size([8])
output.weight torch.Size([2, 8])
output.bias torch.Size([2])


### Automatic Gradient Tracking

During the forward pass:

- PyTorch dynamically builds a computation graph
- Calling `loss.backward()` computes gradients
- Gradients are stored in each parameter’s `.grad`

No manual graph management is required.

In [10]:
x = torch.randn(5, 4)
target = torch.randn(5, 2)

criterion = nn.MSELoss()
output = model(x)
loss = criterion(output, target)

loss.backward()

print("\nGradient computed for output layer:",
      model.output.weight.grad is not None)
model.output.weight.grad


Gradient computed for output layer: True


tensor([[ 0.2028,  0.0260,  0.1950,  0.0106,  0.0785,  0.0601, -0.2208,  0.1239],
        [-0.1871, -0.0307, -0.1326, -0.0527, -0.0651, -0.0787,  0.1771, -0.1404]])

### Device and Type Transfer (`.to()`)

Calling:

    model.to(device)

or

    model.to(dtype)

moves all:

- Parameters
- Buffers
- Submodules

to CPU/GPU/dtype automatically.

In [11]:
import torch

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

dtype = torch.float32

model = model.to(device=device, dtype=dtype)

x: torch.Tensor = torch.randn(5, 4, device=device, dtype=dtype)

print("Model device:", next(model.parameters()).device)
print("Model dtype:", next(model.parameters()).dtype)

Model device: cpu
Model dtype: torch.float32


### Saving & Loading (`state_dict()`)

- `model.state_dict()` returns all parameters recursively
- `model.load_state_dict(...)` restores them

This works across the full module tree.

In [12]:
state_dict = model.state_dict()
torch.save(state_dict, "combined_model.pt")

# `train()` vs `eval()` Mode in PyTorch

PyTorch modules have two main modes: **training mode** and **evaluation mode**.  
Switching between them affects layers that behave differently during training and inference.

---

## `model.train()`

- Sets the model to **training mode**.
- Used when training the model with gradient updates.
- Affects certain layers, such as:

| Layer Type        | Behavior in `train()` Mode                  |
|------------------|--------------------------------------------|
| `Dropout`         | Randomly zeroes some activations           |
| `BatchNorm`       | Updates running statistics (mean/variance) |

- Gradients are computed as usual.

---

## `model.eval()`

- Sets the model to **evaluation (inference) mode**.
- Used when evaluating or deploying the model.
- Affects certain layers:

| Layer Type        | Behavior in `eval()` Mode                   |
|------------------|--------------------------------------------|
| `Dropout`         | Passes all activations through unchanged  |
| `BatchNorm`       | Uses stored running mean/variance         |

- No layers update internal statistics.
- Gradients are usually not required (often used with `torch.no_grad()`).

---

## Key Points

- Always use `model.train()` during training.
- Always use `model.eval()` during evaluation or testing.
- Forgetting to switch can lead to inconsistent results, especially with `Dropout` or `BatchNorm`.



In [13]:
import torch
import torch.nn as nn

# Simple model with Dropout and BatchNorm
class SimpleModel(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.fc1 = nn.Linear(4, 8)
        self.bn = nn.BatchNorm1d(8)
        self.dropout = nn.Dropout(p=0.5)
        self.fc2 = nn.Linear(8, 2)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.fc1(x)
        x = self.bn(x)
        x = torch.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x


model = SimpleModel()
x = torch.randn(5, 4)

# Training mode
model.train()
output_train = model(x)
print("Training mode output:\n", output_train)

# Evaluation mode
model.eval()
with torch.no_grad():
    output_eval = model(x)
print("Evaluation mode output:\n", output_eval)


Training mode output:
 tensor([[-0.0154, -0.1579],
        [-1.6508, -0.8797],
        [-0.6473, -0.8013],
        [ 0.0990, -0.1677],
        [-0.1320, -0.2086]], grad_fn=<AddmmBackward0>)
Evaluation mode output:
 tensor([[ 0.0958, -0.2134],
        [-0.6467, -0.3622],
        [-0.3267, -0.2936],
        [-0.0842, -0.1646],
        [-0.0361, -0.3957]])


# `torch.no_grad()` and `torch.inference_mode()` in PyTorch

When performing inference (evaluating a model without updating parameters), PyTorch provides context managers to **disable gradient tracking**. This saves memory and speeds up computation.

---

## `torch.no_grad()`

- Disables gradient tracking.
- Useful during evaluation or inference.
- Gradients are **not computed**, but autograd still tracks operations for some internal purposes.
- Can be used as a **context manager** or a **function decorator**.

---

## `torch.inference_mode()`

- Introduced in PyTorch 1.9.
- Similar to `no_grad()`, but **more efficient**.
- Completely disables autograd and reduces memory usage.
- Recommended for pure inference pipelines.



In [14]:
import torch
import torch.nn as nn

class SimpleModel(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.fc = nn.Linear(4, 2)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.fc(x)

    # -----------------------------
    # Using torch.no_grad() as method decorator
    # -----------------------------
    @torch.no_grad()
    def forward_no_grad(self, x: torch.Tensor) -> torch.Tensor:
        return self.fc(x)

    # -----------------------------
    # Using torch.inference_mode() as method decorator
    # -----------------------------
    @torch.inference_mode()
    def forward_inference(self, x: torch.Tensor) -> torch.Tensor:
        return self.fc(x)


model = SimpleModel()
x = torch.randn(5, 4)

# -----------------------------
# Call decorated methods
# -----------------------------
output_no_grad_method = model.forward_no_grad(x)
output_infer_method = model.forward_inference(x)

print("Output no_grad method:\n", output_no_grad_method)
print("Output inference_mode method:\n", output_infer_method)

# -----------------------------
# Using context managers
# -----------------------------

with torch.no_grad():
    output_no_grad_cm = model(x)

with torch.inference_mode():
    output_infer_cm = model(x)

print("Output no_grad context manager:\n", output_no_grad_cm)
print("Output inference_mode context manager:\n", output_infer_cm)


Output no_grad method:
 tensor([[-1.0115,  0.1371],
        [-1.1477,  0.8178],
        [-0.6248,  0.2919],
        [-1.0659,  0.2936],
        [ 0.2229, -0.1575]])
Output inference_mode method:
 tensor([[-1.0115,  0.1371],
        [-1.1477,  0.8178],
        [-0.6248,  0.2919],
        [-1.0659,  0.2936],
        [ 0.2229, -0.1575]])
Output no_grad context manager:
 tensor([[-1.0115,  0.1371],
        [-1.1477,  0.8178],
        [-0.6248,  0.2919],
        [-1.0659,  0.2936],
        [ 0.2229, -0.1575]])
Output inference_mode context manager:
 tensor([[-1.0115,  0.1371],
        [-1.1477,  0.8178],
        [-0.6248,  0.2919],
        [-1.0659,  0.2936],
        [ 0.2229, -0.1575]])


# Disabling Gradients with `requires_grad_(False)`

PyTorch provides a convenient method `requires_grad_()` that can **enable or disable gradients in-place** for all parameters of a model or a tensor.

Using:

```python
param.requires_grad_(False)
```

- Sets `requires_grad=False` **in-place** for that parameter.
- This is useful for freezing models during inference or transfer learning.
- Can be applied to an entire model recursively by iterating over its parameters.


In [15]:
model = SimpleModel()

# Disable gradient computation for all parameters using requires_grad_()
for param in model.parameters():
    param.requires_grad_(False)

# Verify
for name, param in model.named_parameters():
    print(f"{name}: requires_grad={param.requires_grad}")

# Forward pass still works
x = torch.randn(5, 4)
output = model(x)
print("Output shape:", output.shape)

fc.weight: requires_grad=False
fc.bias: requires_grad=False
Output shape: torch.Size([5, 2])


# Redefining `train()` and `eval()` in `nn.Module`

PyTorch’s `nn.Module` provides built-in `train(mode: bool = True)` and `eval()` methods to switch between **training** and **evaluation** modes.  

Sometimes, when creating **custom modules or blocks**, you might want to **override these methods** to perform extra actions whenever the mode changes.

---

## Why Override?

- Apply mode-specific logic to sub-blocks or attributes that are not standard layers
- Log or track mode switches
- Automatically modify internal flags or buffers along with training/eval mode

---

## How It Works

- `train(mode: bool = True)` sets `self.training = mode` for the module
- `eval()` is equivalent to `train(False)`
- Default implementation recursively calls `train(mode)` on all submodules
- Overriding allows custom behavior while keeping recursive updates intact


In [16]:
import torch
import torch.nn as nn
from torch import Tensor
from typing import Self

class CustomBlock(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.linear = nn.Linear(4, 4)
        self.dropout = nn.Dropout(p=0.5)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.linear(x)
        x = torch.relu(x)
        x = self.dropout(x)
        return x

    # -----------------------------
    # Override train() method
    # -----------------------------
    def train(self, mode: bool = True) -> Self:
        print(f"CustomBlock set to {'train' if mode else 'eval'} mode")
        super().train(mode)  # Call original method to update submodules
        # Add any custom logic here
        return self

    # -----------------------------
    # Override eval() method
    # -----------------------------
    def eval(self) -> Self:
        print("CustomBlock set to eval mode")
        return super().eval()


# Example usage
model = CustomBlock()
x = torch.randn(2, 4)

# Switch to training mode
model.train()
output_train = model(x)

# Switch to evaluation mode
model.eval()
with torch.no_grad():
    output_eval = model(x)

print("Output training mode:", output_train)
print("Output eval mode:", output_eval)


CustomBlock set to train mode
CustomBlock set to eval mode
CustomBlock set to eval mode
Output training mode: tensor([[1.7513, 0.0000, 0.0000, 0.0000],
        [0.3474, 0.0000, 0.5683, 0.2674]], grad_fn=<MulBackward0>)
Output eval mode: tensor([[0.8756, 0.0000, 1.3273, 1.0028],
        [0.1737, 0.0000, 0.2841, 0.1337]])


# Common Module Aggregators in PyTorch

When building neural networks, it is often useful to group multiple layers or submodules together.  
PyTorch provides several **module aggregators** that help organize layers and blocks. The most common ones are:



## `nn.Sequential`

- Holds modules in a sequential order.
- Executes them **in the order they are added** during the forward pass.
- Ideal for simple **stacked layers** with a single input and output.

**Key points:**

- Forward pass is automatically defined.
- Cannot handle multiple inputs or branching.

In [17]:
seq_model = nn.Sequential(
    nn.Linear(4, 8),
    nn.ReLU(),
    nn.Linear(8, 2)
)

x = torch.randn(5, 4)
output_seq = seq_model(x)
print("nn.Sequential output shape:", output_seq.shape)


nn.Sequential output shape: torch.Size([5, 2])


## `nn.ModuleList`

- Holds a **list of modules**.
- Does **not define a forward pass automatically**.
- Useful when you need to **loop over modules**, or have conditional computation.

**Key points:**

- Modules are registered properly, so parameters are tracked.
- You must define your own `forward()`.

In [18]:
class ModuleListModel(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.layers = nn.ModuleList([
            nn.Linear(4, 8),
            nn.ReLU(),
            nn.Linear(8, 2)
        ])

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        for layer in self.layers:
            x = layer(x)
        return x

ml_model = ModuleListModel()
output_ml = ml_model(x)
print("nn.ModuleList output shape:", output_ml.shape)

nn.ModuleList output shape: torch.Size([5, 2])


## `nn.ModuleDict`

- Holds modules in a **dictionary** with string keys.
- Useful for architectures with **named branches**, **dynamic selection**, or **multi-head outputs**.
- Like `ModuleList`, it does **not define a forward pass**.

In [19]:
class ModuleDictModel(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.branches = nn.ModuleDict({
            "branch1": nn.Linear(4, 8),
            "branch2": nn.Linear(4, 8)
        })
        self.output: nn.Linear = nn.Linear(8, 2)

    def forward(self, x: torch.Tensor, branch_name: str = "branch1") -> torch.Tensor:
        x = self.branches[branch_name](x)
        return self.output(x)

md_model = ModuleDictModel()
output_md = md_model(x, branch_name="branch2")
print("nn.ModuleDict output shape:", output_md.shape)

nn.ModuleDict output shape: torch.Size([5, 2])


## Type hints and Shape Information

For production code, a practical pattern is to use `Tensor` for type hints and document the expected shape next to the signature.

`Tensor` tells us what kind of object is expected, while `[B, C, H, W]` documents the tensor dimensions explicitly.


In [20]:
from torch import Tensor

def normalize_images(x: Tensor) -> Tensor:
    # x: [B, C, H, W]
    return x / 255.0



## Работа на семинаре

### LSTM

![lstm](assets/LSTM.png)

In [ ]:
import torch
import torch.nn as nn
from torch import Tensor
from typing import Optional


class LSTMCell(nn.Module):
    def __init__(self, input_size: int, hidden_size: int) -> None:
        super().__init__()
        self.hidden_size = hidden_size
        # Все 4 гейта (i, f, g, o) считаются одной матрицей: W_i, W_f, W_c, W_o
        self.x2h = nn.Linear(input_size, 4 * hidden_size)
        self.h2h = nn.Linear(hidden_size, 4 * hidden_size)

    def forward(
        self, x: Tensor, state: Optional[tuple[Tensor, Tensor]] = None
    ) -> tuple[Tensor, Tensor]:
        if state is None:
            zeros = x.new_zeros(x.size(0), self.hidden_size)
            state = (zeros, zeros)
        h, c = state

        gates = self.x2h(x) + self.h2h(h)
        i, f, g, o = gates.chunk(4, dim=-1)
        i = torch.sigmoid(i)
        f = torch.sigmoid(f)
        g = torch.tanh(g)
        o = torch.sigmoid(o) 

        c = f * c + i * g
        h = o * torch.tanh(c)
        return h, c


class LSTM(nn.Module):
    def __init__(self, input_size: int, hidden_size: int) -> None:
        super().__init__()
        self.cell = LSTMCell(input_size, hidden_size)

    def forward(
        self, x: Tensor, state: Optional[tuple[Tensor, Tensor]] = None
    ) -> tuple[Tensor, tuple[Tensor, Tensor]]:
        outputs = []
        for t in range(x.size(1)):
            state = self.cell(x[:, t], state)
            outputs.append(state[0])
        return torch.stack(outputs, dim=1), state

torch.manual_seed(0)
B, T, D_in, H = 3, 7, 5, 8
x = torch.randn(B, T, D_in)

my_lstm = LSTM(D_in, H)
ref_lstm = nn.LSTM(D_in, H, batch_first=True)
with torch.no_grad():
    ref_lstm.weight_ih_l0.copy_(my_lstm.cell.x2h.weight)
    ref_lstm.bias_ih_l0.copy_(my_lstm.cell.x2h.bias)
    ref_lstm.weight_hh_l0.copy_(my_lstm.cell.h2h.weight)
    ref_lstm.bias_hh_l0.copy_(my_lstm.cell.h2h.bias)

out, (h, c) = my_lstm(x)
ref_out, (ref_h, ref_c) = ref_lstm(x)

assert out.shape == (B, T, H)
assert torch.allclose(out, ref_out, atol=1e-6), "Выходы LSTM не совпадают с nn.LSTM"
assert torch.allclose(h, ref_h[0], atol=1e-6) and torch.allclose(c, ref_c[0], atol=1e-6)
print("LSTM: все тесты пройдены, output shape:", tuple(out.shape))

LSTM: все тесты пройдены, output shape: (3, 7, 8)


### Inception

![inception](assets/inception.png)

In [ ]:
class ConvReLU(nn.Sequential):
    def __init__(self, in_channels: int, out_channels: int, kernel_size: int) -> None:
        super().__init__(
            nn.Conv2d(in_channels, out_channels, kernel_size, padding=kernel_size // 2),
            nn.ReLU(inplace=True),
        )


class InceptionBlock(nn.Module):
    def __init__(
        self,
        in_channels: int,
        ch1x1: int,
        ch3x3_reduce: int,
        ch3x3: int,
        ch5x5_reduce: int,
        ch5x5: int,
        pool_proj: int,
    ) -> None:
        super().__init__()
        self.branch1 = ConvReLU(in_channels, ch1x1, 1)
        self.branch2 = nn.Sequential(
            ConvReLU(in_channels, ch3x3_reduce, 1),
            ConvReLU(ch3x3_reduce, ch3x3, 3),
        )
        self.branch3 = nn.Sequential(
            ConvReLU(in_channels, ch5x5_reduce, 1),
            ConvReLU(ch5x5_reduce, ch5x5, 5),
        )
        self.branch4 = nn.Sequential(
            nn.MaxPool2d(kernel_size=3, stride=1, padding=1),
            ConvReLU(in_channels, pool_proj, 1),
        )

    def forward(self, x: Tensor) -> Tensor:
        branches = [self.branch1(x), self.branch2(x), self.branch3(x), self.branch4(x)]
        return torch.cat(branches, dim=1)

block = InceptionBlock(192, 64, 96, 128, 16, 32, 32)
x = torch.randn(2, 192, 28, 28)
out = block(x)
assert out.shape == (2, 64 + 128 + 32 + 32, 28, 28)
print("Inception: output shape:", tuple(out.shape))

Inception: output shape: (2, 256, 28, 28)


### SE

![se](assets/SqueezeAndExcite.png)

In [ ]:
class SEBlock(nn.Module):
    def __init__(self, channels: int, reduction: int = 16) -> None:
        super().__init__()
        self.squeeze = nn.AdaptiveAvgPool2d(1)
        self.excitation = nn.Sequential(
            nn.Linear(channels, channels // reduction),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels),
            nn.Sigmoid(),
        )

    def forward(self, x: Tensor) -> Tensor:
        # x: [B, C, H, W]
        b, c = x.shape[:2]
        s = self.squeeze(x).flatten(1)           
        w = self.excitation(s).view(b, c, 1, 1)   
        return x * w                                    



se = SEBlock(64, reduction=16)
x = torch.rand(2, 64, 16, 16) + 0.1
out = se(x)
assert out.shape == x.shape

scale = out / x
assert torch.allclose(scale, scale[..., :1, :1].expand_as(scale), atol=1e-6)
assert ((scale > 0) & (scale < 1)).all()
print("SE: output shape:", tuple(out.shape))

SE: output shape: (2, 64, 16, 16)


### Selective Kernel

![selective](assets/SelectiveKernel.png)


In [ ]:
class SKConv(nn.Module):
    def __init__(
        self,
        channels: int,
        kernel_sizes: tuple[int, ...] = (3, 5),
        reduction: int = 16,
        min_dim: int = 32,
    ) -> None:
        super().__init__()
        d = max(channels // reduction, min_dim)

        self.branches = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(channels, channels, k, padding=k // 2, bias=False),
                nn.BatchNorm2d(channels),
                nn.ReLU(inplace=True),
            )
            for k in kernel_sizes
        ])
        self.fc = nn.Sequential(
            nn.Linear(channels, d, bias=False),
            nn.BatchNorm1d(d),
            nn.ReLU(inplace=True),
        )
        self.fcs = nn.ModuleList([nn.Linear(d, channels) for _ in kernel_sizes])

    def forward(self, x: Tensor) -> Tensor:
        # x: [B, C, H, W]
        feats = torch.stack([branch(x) for branch in self.branches], dim=1)
        u = feats.sum(dim=1)                                               
        s = u.mean(dim=(2, 3))                                          
        z = self.fc(s)                                                    
        logits = torch.stack([fc(z) for fc in self.fcs], dim=1)         
        attn = logits.softmax(dim=1)                                     
        return (feats * attn[..., None, None]).sum(dim=1)        

sk = SKConv(64)
x = torch.randn(4, 64, 16, 16)
out = sk(x)
assert out.shape == x.shape
print("SK: output shape:", tuple(out.shape))

SK: output shape: (4, 64, 16, 16)


### PatchMerger

![patchmerger](assets/PatchMerger.png)


In [ ]:
class PatchMerger(nn.Module):
    def __init__(self, dim: int, num_tokens_out: int) -> None:
        super().__init__()
        self.norm = nn.LayerNorm(dim)
        # Learned Weights M x D
        self.queries = nn.Parameter(torch.randn(num_tokens_out, dim) * dim ** -0.5)

    def forward(self, x: Tensor) -> Tensor:
        # x: [B, N, D] -> [B, M, D]
        x = self.norm(x)
        sim = self.queries @ x.transpose(-1, -2) 
        attn = sim.softmax(dim=-1)                 
        return attn @ x        

merger = PatchMerger(dim=128, num_tokens_out=8)
x = torch.randn(2, 196, 128)
out = merger(x)
assert out.shape == (2, 8, 128)
print("PatchMerger: output shape:", tuple(out.shape))

PatchMerger: output shape: (2, 8, 128)


## Домашнее задание

## Задание 1, реализуйте требуемые блоки

## ResNet Block (0.1 балл)

![Resnet](assets/ResBlock.png)

https://arxiv.org/pdf/1512.03385

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, stride: int = 1) -> None:
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)

        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels),
            )
        else:
            self.shortcut = nn.Identity()

    def forward(self, x: Tensor) -> Tensor:
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return self.relu(out + self.shortcut(x))

x = torch.randn(2, 64, 32, 32)

block = ResidualBlock(64, 64)
assert block(x).shape == (2, 64, 32, 32)

block_down = ResidualBlock(64, 128, stride=2)
assert block_down(x).shape == (2, 128, 16, 16)

nn.init.zeros_(block.bn2.weight)
assert torch.allclose(block(x), torch.relu(x))
print("ResidualBlock: все тесты пройдены")

ResidualBlock: все тесты пройдены


## Depthwise Separable Convolution (0.1 балл)
![DepthWiseConv](assets/DepthWiseConv.png)

https://arxiv.org/pdf/1610.02357

In [ ]:
import torch.nn.functional as F


class SeparableConv2d(nn.Module):
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        kernel_size: int = 3,
        stride: int = 1,
        padding: int = 1,
        dilation: int = 1,
        bias: bool = False,
    ) -> None:
        super().__init__()
        self.depthwise = nn.Conv2d(
            in_channels, in_channels, kernel_size,
            stride=stride, padding=padding, dilation=dilation,
            groups=in_channels, bias=bias,
        )
        self.pointwise = nn.Conv2d(in_channels, out_channels, 1, bias=bias)

    def forward(self, x: Tensor) -> Tensor:
        return self.pointwise(self.depthwise(x))

M, N, k = 32, 64, 3
x = torch.randn(2, M, 16, 16)
sep = SeparableConv2d(M, N, k)
out = sep(x)
assert out.shape == (2, N, 16, 16)

dw = sep.depthwise.weight[:, 0]               
pw = sep.pointwise.weight[:, :, 0, 0]               
full_weight = pw[:, :, None, None] * dw[None]        
assert torch.allclose(out, F.conv2d(x, full_weight, padding=1), atol=1e-5)

n_sep = sum(p.numel() for p in sep.parameters())
n_full = nn.Conv2d(M, N, k, bias=False).weight.numel()
print(f"SeparableConv2d: {n_sep} параметров против {n_full} у обычной свёртки")

SeparableConv2d: 2336 параметров против 18432 у обычной свёртки


## Vanilla Attention (0.1 балл)

Let:

$$
\text{query} \in \mathbb{R}^{B \times d} \\
\text{key} \in \mathbb{R}^{B \times L \times d}
$$

---

### Alignment Scores

$$
\text{score} = \text{key} \cdot (W_\text{align} \, \text{query})^T \\
\text{score} \in \mathbb{R}^{B \times L}
$$

---

### Attention Weights

$$
\text{att} = \text{softmax}(\text{score}, \text{dim}=1) \\
\text{att} \in \mathbb{R}^{B \times L}
$$

---

### Context Vector

$$
\text{context} = \sum_{i=1}^{L} \text{att}_i \cdot \text{key}_i \\
\text{context} \in \mathbb{R}^{B \times d}
$$

---

### Output

$$
\text{out} = \tanh(W_\text{value} \, \text{context} + W_\text{query} \, \text{query}) \\
\text{out} \in \mathbb{R}^{B \times d}
$$



https://arxiv.org/abs/1409.0473


https://arxiv.org/abs/1508.04025

In [ ]:
from typing import Optional
import torch
from torch import nn
import numpy as np

class VanillaAttention(nn.Module):
    def __init__(self, dim: int) -> None:
        super().__init__()
        self.W_align = nn.Linear(dim, dim, bias=False)
        self.W_value = nn.Linear(dim, dim, bias=False)
        self.W_query = nn.Linear(dim, dim, bias=False)

    def forward(
        self, query: torch.Tensor, key: torch.Tensor, mask: Optional[torch.Tensor] = None
    ) -> tuple[torch.Tensor, torch.Tensor]:
        score = torch.bmm(key, self.W_align(query).unsqueeze(-1)).squeeze(-1)  
        if mask is not None:
            score = score.masked_fill(~mask, float("-inf"))
        att = torch.softmax(score, dim=1)                                
        context = torch.bmm(att.unsqueeze(1), key).squeeze(1)              
        out = torch.tanh(self.W_value(context) + self.W_query(query))  
        return out, att

torch.manual_seed(0)
B, L, d = 4, 10, 16
query = torch.randn(B, d)
key = torch.randn(B, L, d)

attn = VanillaAttention(d)
out, att = attn(query, key)
assert out.shape == (B, d) and att.shape == (B, L)
assert torch.allclose(att.sum(dim=1), torch.ones(B))

q, k = query.numpy(), key.numpy()
W_a, W_v, W_q = (m.weight.detach().numpy() for m in (attn.W_align, attn.W_value, attn.W_query))
score = np.einsum("bld,bd->bl", k, q @ W_a.T)
att_np = np.exp(score - score.max(1, keepdims=True))
att_np /= att_np.sum(1, keepdims=True)
context = np.einsum("bl,bld->bd", att_np, k)
out_np = np.tanh(context @ W_v.T + q @ W_q.T)
assert np.allclose(out.detach().numpy(), out_np, atol=1e-5)

mask = torch.ones(B, L, dtype=torch.bool)
mask[:, -3:] = False
_, att_masked = attn(query, key, mask)
assert (att_masked[:, -3:] == 0).all()
print("VanillaAttention: все тесты пройдены")

VanillaAttention: все тесты пройдены


## Dot Product Attention (0.1 балл)

$$
Q \in \mathbb{R}^{B \times L_q \times d_k} \\
K \in \mathbb{R}^{B \times L_k \times d_k} \\
V \in \mathbb{R}^{B \times L_k \times d_k}
$$

$$
S = \frac{Q K^T}{\sqrt{d_k}}
$$

$$
\text{Attention}(Q, K, V) = \text{softmax}(S, \text{dim}=-1) \, V
$$



https://arxiv.org/abs/1706.03762


In [ ]:
import math


class ScaledDotProductAttention(nn.Module):
    def __init__(self, dropout: float = 0.0) -> None:
        super().__init__()
        self.dropout = nn.Dropout(dropout)

    def forward(
        self, q: Tensor, k: Tensor, v: Tensor, mask: Optional[Tensor] = None
    ) -> tuple[Tensor, Tensor]:
        scores = q @ k.transpose(-2, -1) / math.sqrt(q.size(-1))  # [..., L_q, L_k]
        if mask is not None:
            scores = scores.masked_fill(~mask, float("-inf"))
        attn = self.dropout(scores.softmax(dim=-1))
        return attn @ v, attn

torch.manual_seed(0)
q = torch.randn(2, 5, 16)
k = torch.randn(2, 7, 16)
v = torch.randn(2, 7, 16)

sdpa = ScaledDotProductAttention()
out, attn = sdpa(q, k, v)
assert out.shape == (2, 5, 16) and attn.shape == (2, 5, 7)
assert torch.allclose(out, F.scaled_dot_product_attention(q, k, v), atol=1e-6)

causal = torch.ones(5, 5, dtype=torch.bool).tril()
out_causal, _ = sdpa(q, q, q, causal)
assert torch.allclose(out_causal, F.scaled_dot_product_attention(q, q, q, is_causal=True), atol=1e-6)
print("ScaledDotProductAttention: все тесты пройдены")

ScaledDotProductAttention: все тесты пройдены


## Multihead Attention (0.1 балл)

![MultiheadAttention](assets/MultiheadAttention.webp)

https://arxiv.org/abs/1706.03762


In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, embed_dim: int, num_heads: int, dropout: float = 0.0, bias: bool = True) -> None:
        super().__init__()
        assert embed_dim % num_heads == 0, "embed_dim должен делиться на num_heads"
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads

        self.q_proj = nn.Linear(embed_dim, embed_dim, bias=bias)
        self.k_proj = nn.Linear(embed_dim, embed_dim, bias=bias)
        self.v_proj = nn.Linear(embed_dim, embed_dim, bias=bias)
        self.attention = ScaledDotProductAttention(dropout)
        self.out_proj = nn.Linear(embed_dim, embed_dim, bias=bias)

    def _split_heads(self, x: Tensor) -> Tensor:
        b, l, _ = x.shape
        return x.view(b, l, self.num_heads, self.head_dim).transpose(1, 2)

    def forward(
        self,
        query: Tensor,
        key: Optional[Tensor] = None,
        value: Optional[Tensor] = None,
        mask: Optional[Tensor] = None,
    ) -> tuple[Tensor, Tensor]:
        key = query if key is None else key
        value = key if value is None else value

        q = self._split_heads(self.q_proj(query))
        k = self._split_heads(self.k_proj(key))
        v = self._split_heads(self.v_proj(value))
        if mask is not None and mask.dim() == 3:
            mask = mask.unsqueeze(1)

        out, attn = self.attention(q, k, v, mask)
        b, _, l_q, _ = out.shape
        out = out.transpose(1, 2).reshape(b, l_q, -1) 
        return self.out_proj(out), attn

torch.manual_seed(0)
B, L, D, h = 2, 6, 32, 4
x = torch.randn(B, L, D)
memory = torch.randn(B, 9, D)

mha = MultiHeadAttention(D, h)
ref = nn.MultiheadAttention(D, h, batch_first=True)
with torch.no_grad():
    ref.in_proj_weight.copy_(torch.cat([mha.q_proj.weight, mha.k_proj.weight, mha.v_proj.weight]))
    ref.in_proj_bias.copy_(torch.cat([mha.q_proj.bias, mha.k_proj.bias, mha.v_proj.bias]))
    ref.out_proj.weight.copy_(mha.out_proj.weight)
    ref.out_proj.bias.copy_(mha.out_proj.bias)

out, attn = mha(x)
ref_out, ref_attn = ref(x, x, x)
assert out.shape == (B, L, D) and attn.shape == (B, h, L, L)
assert torch.allclose(out, ref_out, atol=1e-5)
assert torch.allclose(attn.mean(dim=1), ref_attn, atol=1e-6)

out_x, _ = mha(x, memory)
assert torch.allclose(out_x, ref(x, memory, memory)[0], atol=1e-5)
causal = torch.ones(L, L, dtype=torch.bool).tril()
assert torch.allclose(mha(x, mask=causal)[0], ref(x, x, x, attn_mask=~causal)[0], atol=1e-5)
print("MultiHeadAttention: все тесты пройдены")

MultiHeadAttention: все тесты пройдены


## Transformer Encoder Layer (0.1 балл)


![Transformer Encoder Layer](assets/TransformerEncoder.png)


https://arxiv.org/abs/1706.03762

In [ ]:
class TransformerEncoderLayer(nn.Module):
    def __init__(self, embed_dim: int, num_heads: int, mlp_dim: int, dropout: float = 0.0) -> None:
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn = MultiHeadAttention(embed_dim, num_heads, dropout)
        self.attn_dropout = nn.Dropout(dropout)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, mlp_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_dim, embed_dim),
            nn.Dropout(dropout),
        )

    def forward(self, x: Tensor, mask: Optional[Tensor] = None) -> Tensor:
        h = self.norm1(x)
        attn_out, _ = self.attn(h, h, h, mask)
        x = x + self.attn_dropout(attn_out)
        x = x + self.mlp(self.norm2(x))
        return x


torch.manual_seed(0)
B, N, D, h, mlp_dim = 2, 10, 64, 8, 128
x = torch.randn(B, N, D)

layer = TransformerEncoderLayer(D, h, mlp_dim).eval()
ref = nn.TransformerEncoderLayer(
    D, h, mlp_dim, dropout=0.0, activation="gelu", batch_first=True, norm_first=True
).eval()
with torch.no_grad():
    a = layer.attn
    ref.self_attn.in_proj_weight.copy_(torch.cat([a.q_proj.weight, a.k_proj.weight, a.v_proj.weight]))
    ref.self_attn.in_proj_bias.copy_(torch.cat([a.q_proj.bias, a.k_proj.bias, a.v_proj.bias]))
    ref.self_attn.out_proj.load_state_dict(a.out_proj.state_dict())
    ref.linear1.load_state_dict(layer.mlp[0].state_dict())
    ref.linear2.load_state_dict(layer.mlp[3].state_dict())
    ref.norm1.load_state_dict(layer.norm1.state_dict())
    ref.norm2.load_state_dict(layer.norm2.state_dict())

out = layer(x)
assert out.shape == (B, N, D)
assert torch.allclose(out, ref(x), atol=1e-5)
print("TransformerEncoderLayer: все тесты пройдены")

TransformerEncoderLayer: все тесты пройдены


## MLP Mixer (0.1 балл)


![MLPMixer](assets/MLPMixer.png)


https://arxiv.org/abs/2105.01601

In [ ]:
class MlpBlock(nn.Module):
    def __init__(self, dim: int, hidden_dim: int, dropout: float = 0.0) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, dim),
            nn.Dropout(dropout),
        )

    def forward(self, x: Tensor) -> Tensor:
        return self.net(x)


class MLPMixerBlock(nn.Module):
    def __init__(
        self,
        num_patches: int,
        hidden_dim: int,
        tokens_mlp_dim: int,
        channels_mlp_dim: int,
        dropout: float = 0.0,
    ) -> None:
        super().__init__()
        self.norm1 = nn.LayerNorm(hidden_dim)
        self.token_mlp = MlpBlock(num_patches, tokens_mlp_dim, dropout)    
        self.norm2 = nn.LayerNorm(hidden_dim)
        self.channel_mlp = MlpBlock(hidden_dim, channels_mlp_dim, dropout) 

    def forward(self, x: Tensor) -> Tensor:
        # x: [B, N, C] (N патчей, C каналов)
        y = self.norm1(x).transpose(1, 2)        
        x = x + self.token_mlp(y).transpose(1, 2)    
        x = x + self.channel_mlp(self.norm2(x))      
        return x


# ----------------------------
# Тестирование
# ----------------------------
torch.manual_seed(0)
B, N, C = 2, 49, 64
block = MLPMixerBlock(num_patches=N, hidden_dim=C, tokens_mlp_dim=32, channels_mlp_dim=256)
x = torch.randn(B, N, C)
out = block(x)
assert out.shape == (B, N, C)

# Token-mixing: изменение одного патча влияет на выходы всех остальных патчей
x2 = x.clone()
x2[:, 0] += 1.0
diff = (block(x2) - out).abs().sum(dim=-1)   # [B, N]
assert (diff[:, 1:] > 0).all()
print("MLPMixerBlock: output shape:", tuple(out.shape))

MLPMixerBlock: output shape: (2, 49, 64)


## ConvMixer (0.1 балл)

![ConvMixer](assets/ConvMixer.png)


https://arxiv.org/abs/2201.09792

In [33]:
class Residual(nn.Module):
    def __init__(self, fn: nn.Module) -> None:
        super().__init__()
        self.fn = fn

    def forward(self, x: Tensor) -> Tensor:
        return x + self.fn(x)


class ConvMixerLayer(nn.Sequential):
    def __init__(self, dim: int, kernel_size: int, activation: type[nn.Module] = nn.GELU) -> None:
        super().__init__(
            # Depthwise Conv -> Act -> BN, обёрнутые в residual connection (смешивание по пространству)
            Residual(nn.Sequential(
                nn.Conv2d(dim, dim, kernel_size, groups=dim, padding="same"),
                activation(),
                nn.BatchNorm2d(dim),
            )),
            # Pointwise Conv -> Act -> BN (смешивание по каналам)
            nn.Conv2d(dim, dim, kernel_size=1),
            activation(),
            nn.BatchNorm2d(dim),
        )


class ConvMixer(nn.Module):

    def __init__(
        self,
        dim: int,
        depth: int,
        kernel_size: int = 9,
        patch_size: int = 7,
        in_channels: int = 3,
        num_classes: int = 1000,
        activation: type[nn.Module] = nn.GELU,
    ) -> None:
        super().__init__()
        # Patch embedding: свёртка p x p со stride p
        self.patch_embed = nn.Sequential(
            nn.Conv2d(in_channels, dim, kernel_size=patch_size, stride=patch_size),
            activation(),
            nn.BatchNorm2d(dim),
        )
        self.layers = nn.Sequential(*[ConvMixerLayer(dim, kernel_size, activation) for _ in range(depth)])
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(dim, num_classes),
        )

    def forward(self, x: Tensor) -> Tensor:
        # x: [B, C_in, H, W] -> logits: [B, num_classes]
        x = self.patch_embed(x)   # [B, dim, H / p, W / p]
        x = self.layers(x)        # [B, dim, H / p, W / p]
        return self.head(x)


# ----------------------------
# Тестирование
# ----------------------------
torch.manual_seed(0)
model = ConvMixer(dim=128, depth=4, kernel_size=5, patch_size=2, num_classes=10)
x = torch.randn(2, 3, 32, 32)
logits = model(x)
assert logits.shape == (2, 10)

layer = ConvMixerLayer(dim=128, kernel_size=5)
feat = torch.randn(2, 128, 16, 16)
assert layer(feat).shape == feat.shape

logits.sum().backward()
assert all(p.grad is not None for p in model.parameters())
print("ConvMixer: logits shape:", tuple(logits.shape))
print("Параметров:", sum(p.numel() for p in model.parameters()))

ConvMixer: logits shape: (2, 10)
Параметров: 84618


## Вопрос (0.2 балла)

Объясните, почему MLPMixer, ConvMixer может работать почти так же эффективно, как обычный Multihead Attention.

Напишите формулу, связывающую Multihead Attention, ConvMixer и MLPMixer

Опишите преимущества и недостатки между ConvMixer, MLPMixer и Multihead Attention

---

**Ответ:**

Все три модели устроены по одной схеме. Изображение режется на патчи. Затем блоки чередуют два шага: смешивание токенов (обмен информацией между патчами) и смешивание каналов (MLP для каждого патча). Каждый шаг обёрнут в нормализацию и остаточную связь:

$$
X' = X + \text{TokenMixer}(\text{Norm}(X)), \qquad
Y = X' + \text{ChannelMLP}(\text{Norm}(X'))
$$

Channel-MLP во всех трёх моделях одинаковый, отличается только token mixer. Эксперименты MLP-Mixer, ConvMixer и PoolFormer показывают, что качество в основном даёт сама схема, а не механизм внимания. Внимание — лишь один из способов смешивать токены.

**Общая формула.** Пусть $X \in \mathbb{R}^{N \times C}$ — это N токенов по C каналов. Каждый token mixer умножает токены на матрицу смешивания $A \in \mathbb{R}^{N \times N}$:

$$
\text{TokenMixer}(X)_{:,c} = A_c \, X_{:,c}
$$

- **Multihead Attention:** $A^{(h)} = \text{softmax}\big(X W_Q^{(h)} (X W_K^{(h)})^T / \sqrt{d_k}\big)$. Матрица плотная и вычисляется заново для каждого входа.
- **MLP-Mixer:** $A = W_{token}$. Матрица плотная, но после обучения фиксирована и общая для всех каналов.
- **ConvMixer:** $A_c = \text{Toeplitz}(w_c)$, ленточная матрица depthwise-свёртки $k \times k$. Она фиксирована, локальна и своя для каждого канала.

Если в механизме внимания сделать $A$ независимой от входа, получится MLP-Mixer. Если вдобавок сделать $A$ локальной с общими весами по позициям, получится ConvMixer.

**Преимущества и недостатки.**

- **Multihead Attention.** Плюсы: глобальный обзор с первого слоя, веса подстраиваются под вход, работает с любой длиной последовательности и с cross-attention. Минусы: время и память $O(N^2)$, нужно много данных.
- **MLP-Mixer.** Плюсы: глобальный обзор, только матричные умножения, очень простая реализация. Минусы: веса не зависят от входа, число патчей $N$ зашито в размер слоя, нужно ещё больше данных.
- **ConvMixer.** Плюсы: сложность $O(N k^2)$, мало параметров, любой размер изображения, хорошо учится на небольших датасетах. Минусы: слой видит только соседей $k \times k$, поэтому нужны большие ядра или глубина. Depthwise-свёртки с большими ядрами работают на GPU медленно.